#  Model Architecture





---
##  Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/prajwl7676/mini-clip-DLAI.git

%cd /content/mini-clip-DLAI


!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '/content/mini-clip-DLAI')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from transformers import DistilBertModel, DistilBertTokenizer
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

---
##  Image Encoder with projection head


In [ ]:
EMBEDDING_DIM = 256   # shared dimensionality for both encoders

class ImageEncoder(nn.Module):
    def __init__(self, embedding_dim=EMBEDDING_DIM, freeze_layers=True):
        super().__init__()

        resnet = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        # backbone output: (B, 2048, 1, 1)

        self.projection = nn.Sequential(
            nn.Linear(2048, embedding_dim, bias=False),
            nn.LayerNorm(embedding_dim),
        )

        if freeze_layers:
            for layer in [resnet.conv1, resnet.bn1, resnet.layer1,
                          resnet.layer2, resnet.layer3]:
                for param in layer.parameters():
                    param.requires_grad = False

    def forward(self, images):
        features   = self.backbone(images).flatten(start_dim=1)  # (B, 2048)
        embeddings = self.projection(features)                    # (B, embedding_dim)
        return F.normalize(embeddings, p=2, dim=-1)               # L2-normalised


image_encoder = ImageEncoder().to(device)
print('ImageEncoder created.')

---
## Test the Image Encoder with a fake batch


In [ ]:
BATCH_SIZE = 8

fake_images = torch.randn(BATCH_SIZE, 3, 224, 224).to(device)

with torch.no_grad():
    image_embeddings = image_encoder(fake_images)

print(f'Input  shape : {fake_images.shape}')         # (8, 3, 224, 224)
print(f'Output shape : {image_embeddings.shape}')    # (8, 256)
print(f'L2 norms (should all be ~1.0): {image_embeddings.norm(dim=-1).tolist()}')

assert image_embeddings.shape == (BATCH_SIZE, EMBEDDING_DIM), 'Wrong output shape!'
assert torch.allclose(image_embeddings.norm(dim=-1), torch.ones(BATCH_SIZE).to(device), atol=1e-5), 'Not L2-normalised!'
print('\nAll image encoder checks passed.')

---
##  Image Encoder parameter breakdown


In [ ]:
def count_params(model, name='Model'):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    print(f'{name}')
    print(f'  Total params     : {total:>12,}')
    print(f'  Trainable params : {trainable:>12,}  ({100*trainable/total:.1f}%)')
    print(f'  Frozen params    : {frozen:>12,}  ({100*frozen/total:.1f}%)')
    return trainable

count_params(image_encoder, 'ImageEncoder')

---
## Text Encoder with projection head

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, embedding_dim=EMBEDDING_DIM, model_name='distilbert-base-uncased',freeze_layers=True):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained(model_name)

        if freeze_layers:
            # 1. Freeze early transformer layers (0 and 1 out of 6)
            for layer in self.bert.transformer.layer[:2]:
                for param in layer.parameters():
                    param.requires_grad = False

            # 2. Also freeze the heavy text embeddings lookup table
            for param in self.bert.embeddings.parameters():
                param.requires_grad = False

        # DistilBERT hidden size = 768
        self.projection = nn.Sequential(
            nn.Linear(768, embedding_dim, bias=False),
            nn.LayerNorm(embedding_dim),
        )

    def forward(self, input_ids, attention_mask):
        output     = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.last_hidden_state[:, 0, :]   # [CLS] token: (B, 768)
        embeddings = self.projection(cls_output)           # (B, embedding_dim)
        return F.normalize(embeddings, p=2, dim=-1)


text_encoder = TextEncoder().to(device)
print('TextEncoder created.')

---
##  Test the Text Encoder with fake tokens

In [ ]:
MAX_LENGTH = 77

# Vocabulary size of DistilBERT = 30,522
fake_input_ids      = torch.randint(0, 30522, (BATCH_SIZE, MAX_LENGTH)).to(device)
fake_attention_mask = torch.ones(BATCH_SIZE, MAX_LENGTH, dtype=torch.long).to(device)

with torch.no_grad():
    text_embeddings = text_encoder(fake_input_ids, fake_attention_mask)

print(f'input_ids shape  : {fake_input_ids.shape}')     # (8, 77)
print(f'Output shape     : {text_embeddings.shape}')    # (8, 256)
print(f'L2 norms         : {text_embeddings.norm(dim=-1).tolist()}')

assert text_embeddings.shape == (BATCH_SIZE, EMBEDDING_DIM)
print('\nAll text encoder checks passed.')

---
##  CLIPModel


In [ ]:
class CLIPModel(nn.Module):
    def __init__(self, embedding_dim=EMBEDDING_DIM, temperature_init=0.07):
        super().__init__()
        self.image_encoder  = ImageEncoder(embedding_dim)
        self.text_encoder   = TextEncoder(embedding_dim)
        # Store log(τ) so τ = exp(log_τ) is always positive
        self.log_temperature = nn.Parameter(torch.tensor(temperature_init).log())

    @property
    def temperature(self):
        return self.log_temperature.exp()

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, input_ids, attention_mask):
        return self.text_encoder(input_ids, attention_mask)

    def forward(self, images, input_ids, attention_mask):
        img_emb  = self.image_encoder(images)
        txt_emb  = self.text_encoder(input_ids, attention_mask)
        return img_emb, txt_emb, self.temperature


model = CLIPModel().to(device)
print('CLIPModel created.')
print(f'Initial temperature τ = {model.temperature.item():.4f}')

---
##  Full end-to-end shape test

In [ ]:
with torch.no_grad():
    img_emb, txt_emb, tau = model(fake_images, fake_input_ids, fake_attention_mask)

print('=== CLIPModel forward pass ===')
print(f'Image embeddings : {img_emb.shape}')   # (8, 256)
print(f'Text  embeddings : {txt_emb.shape}')   # (8, 256)
print(f'Temperature τ    : {tau.item():.4f}')

sim_matrix = img_emb @ txt_emb.T
print(f'Similarity matrix: {sim_matrix.shape}')  # (8, 8)

# Sanity: diagonal should be the matched pairs
print(f'Diagonal values (matched pairs)  : {sim_matrix.diag().tolist()}')

assert img_emb.shape == (BATCH_SIZE, EMBEDDING_DIM)
assert txt_emb.shape == (BATCH_SIZE, EMBEDDING_DIM)
assert sim_matrix.shape == (BATCH_SIZE, BATCH_SIZE)
print('\nAll CLIPModel checks passed.')

---
## Full parameter count

In [ ]:
print('=' * 50)
img_t = count_params(model.image_encoder, 'ImageEncoder')
print()
txt_t = count_params(model.text_encoder,  'TextEncoder')
print()
total_t = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_a = sum(p.numel() for p in model.parameters())
print('=' * 50)
print(f'CLIPModel total')
print(f'  All params       : {total_a:>12,}')
print(f'  Trainable params : {total_t:>12,}  ({100*total_t/total_a:.1f}%)')
print(f'  Approx GPU memory: ~{total_t * 4 / 1e6:.0f} MB (float32)')

---
##  Visualise similarity matrix on real data



In [ ]:
from src.dataset import FlickrDataset, get_transform, build_loaders
from torch.utils.data import DataLoader

dataset_hf = load_dataset('nlphuji/flickr30k', split='test[:200]')
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

viz_ds     = FlickrDataset(dataset_hf, tokenizer, get_transform(train=False))
viz_loader = DataLoader(viz_ds, batch_size=8, shuffle=False)

images_batch, tokens_batch = next(iter(viz_loader))
images_batch = images_batch.to(device)
input_ids    = tokens_batch['input_ids'].to(device)
attn_mask    = tokens_batch['attention_mask'].to(device)

print(f'Real batch loaded: {images_batch.shape}')

In [ ]:
# Compute similarity matrix with untrained model
model.eval()
with torch.no_grad():
    img_emb, txt_emb, tau = model(images_batch, input_ids, attn_mask)
    sim = (img_emb @ txt_emb.T).cpu().numpy()   # (8, 8)

# Get captions for labels
captions = [dataset_hf[i]['caption'][0][:30] + '...' for i in range(8)]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim, cmap='RdYlGn', vmin=-0.3, vmax=0.3)

ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels([f'T{i}' for i in range(8)], rotation=45, ha='right', fontsize=9)
ax.set_yticklabels([f'I{i}' for i in range(8)], fontsize=9)
ax.set_xlabel('Text embeddings')
ax.set_ylabel('Image embeddings')
ax.set_title('Similarity matrix — BEFORE training\n(diagonal = matched pairs)')

# Highlight diagonal
for i in range(8):
    ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False,
                                edgecolor='navy', lw=2))
    ax.text(i, i, f'{sim[i,i]:.2f}', ha='center', va='center',
            fontsize=9, fontweight='bold', color='navy')

plt.colorbar(im, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.savefig('similarity_matrix_before_training.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved similarity_matrix_before_training.png')
print()
print('NOTE: Before training, the diagonal has no clear pattern.')
print('After training, the diagonal should be the brightest row in each column.')

---
## Confirm src/model.py import

In [ ]:

from src.model import CLIPModel, count_parameters

model_from_src = CLIPModel(embedding_dim=256).to(device)

# Verify forward pass
with torch.no_grad():
    ie, te, t = model_from_src(fake_images, fake_input_ids, fake_attention_mask)

assert ie.shape == (BATCH_SIZE, 256)
assert te.shape == (BATCH_SIZE, 256)

counts = count_parameters(model_from_src)
print('src.model import works correctly.')
print(f'Trainable params : {counts["trainable"]:,}  ({counts["trainable_pct"]}%)')